In [ ]:
# BLOCO 1: Instalações iniciais

# 1. Downgrade preventivo do Numpy (para evitar conflitos com AV)
!pip install "numpy<2.0"

# 2. Instala AV (biblioteca de decodificação) na versão estável (evita o Silent Hang do MP3)
!pip install "av==15.0.0"

print("\npc_name-------------------------------------------------------------------------")
print("✅ INSTALAÇÕES INICIAIS CONCLUÍDAS.")
print("🔴 AÇÃO CRÍTICA: Você DEVE REINICIAR o ambiente agora para estabilizar o numpy.")
print("Vá em Runtime (Ambiente de Execução) > Restart session (Reiniciar Sessão) ou clique no botão de reinício.")
print("-------------------------------------------------------------------------")

In [ ]:
# BLOCO 2: Instalações Finais (Execute SOMENTE APÓS o reinício do Bloco 1)

# 1. PyTorch + TorchVision
#    A versão 2.3.1 é compatível com o ambiente CUDA 11.8 da GPU T4 do Colab.
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118

# 2. WhisperX e dependências
!pip install git+https://github.com/m-bain/whisperX.git --no-deps
!pip install ffmpeg-python pandas scipy "pyannote.audio==3.3.2" ctranslate2 faster-whisper

# 3. Integrações externas
!pip install --upgrade notion-client

print("\npc_name-------------------------------------------------------------------------")
print("\npc_name✅ INSTALAÇÃO E CONFIGURAÇÃO COMPLETAS. O ambiente está pronto.")
print("🔴 AÇÃO CRÍTICA: Você DEVE REINICIAR o ambiente agora.")
print("Vá em Runtime (Ambiente de Execução) > Restart session (Reiniciar Sessão) ou clique no botão de reinício.")
print("-------------------------------------------------------------------------")

In [ ]:
# BLOCO 3: Montagem do Drive e Configurações da Sessão
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

# ----------------------------------------------------------------------
# EDITE APENAS ESTAS VARIÁVEIS PARA CADA NOVA SESSÃO:
# ----------------------------------------------------------------------

ID_MESA = "ID"
SESSION_NUMBER = "60"
SESSION_DATE = "2026-03-21"  # Formato YYYY-MM-DD
FILE_PART = ""          # Deixe como "" para arquivo único
FORCE_UPDATE = False  # True para puxar mudanças do repo

# ----------------------------------------------------------------------

SESSION_ID = f"{ID_MESA}{SESSION_NUMBER}"
PROJECT_FOLDER = "/content/drive/MyDrive/rpg-transcription"

# ----------------------------------------------------------------------
# CLONAGEM / ATUALIZAÇÃO DO REPOSITÓRIO
# ----------------------------------------------------------------------

if not os.path.exists('rpg_transcription'):
    !git clone https://github.com/ThaisMosken/rpg_transcription.git
elif FORCE_UPDATE:
    !git -C rpg_transcription pull

sys.path.append('/content/rpg_transcription')

# ----------------------------------------------------------------------
# DEFINE O GLOSSÁRIO
# ----------------------------------------------------------------------

from utils.glossary_manager import GlossaryManager

manager = GlossaryManager(ID_MESA)
context_glossary = manager.get_full_glossary()

GLOSSARY = [
    line.strip("- *").strip()
    for line in context_glossary.split('\npc_name')
    if line.strip() and not line.startswith('#')
]

print(f"✅ Glossário para '{ID_MESA}' carregado com sucesso!")

# ----------------------------------------------------------------------
# VARIÁVEIS DERIVADAS (Não precisam ser editadas)
# ----------------------------------------------------------------------

if FILE_PART:
    INPUT_FILE = os.path.join(PROJECT_FOLDER, f"{SESSION_ID}_parte_{FILE_PART}.wav")
    OUTPUT_TXT_FILE = os.path.join(PROJECT_FOLDER, f"{SESSION_ID}_transcricao_final_parte_{FILE_PART}.txt")
else:
    INPUT_FILE = os.path.join(PROJECT_FOLDER, f"{SESSION_ID}w.wav")
    OUTPUT_TXT_FILE = os.path.join(PROJECT_FOLDER, f"{SESSION_ID}_transcricao_final.txt")

CHRONICLE_OUTPUT_FILE = os.path.join(PROJECT_FOLDER, f"{SESSION_ID}_cronica.md")

print("Configuração de sessão concluída:")
print(f"  ID da Sessão: {SESSION_ID}")
print(f"  Entrada WAV: {INPUT_FILE}")
print(f"  Saída TXT: {OUTPUT_TXT_FILE}")
print(f"  Saída Crônica: {CHRONICLE_OUTPUT_FILE}")

In [ ]:
# BLOCO 4: Execução da Transcrição
from src.transcription_processor import execute_transcription

execute_transcription(
    input_file=INPUT_FILE,
    output_file=OUTPUT_TXT_FILE,
    glossary_names=GLOSSARY,
    device="cuda",        # "cuda" ou "cpu"
    model_precision="float16", # "int8_float16" ou "float16"
    model_name="large-v2"
)

In [ ]:
# BLOCO 5: Geração da Crônica
from google.colab import userdata
from src.chronicler import generate_gemini_chronicle

try:
    CHAVE_GEMINI = userdata.get('GEMINI_API_KEY')
except:
    CHAVE_GEMINI = None
    print("ERRO: Secret 'GEMINI_API_KEY' não encontrado.")

# Define o model
MODELO_ESCOLHIDO = 'gemini-2.5-flash'

# Define o template a ser utilizado
# Utilize template_sessao0_v1 para sessão 0 e template_cronica_v1 para sessões normais
TEMPLATE = 'template_cronica_v1'

# Executa a função importada
if CHAVE_GEMINI:
    generate_gemini_chronicle(
        api_key=CHAVE_GEMINI,
        transcription_path=OUTPUT_TXT_FILE,
        output_path=CHRONICLE_OUTPUT_FILE,
        context_glossary=context_glossary,
        model=MODELO_ESCOLHIDO,
        template=TEMPLATE
    )

In [ ]:
# BLOCO 6: Publicação no Notion
import json
from utils.notion_publisher import NotionPublisher
from google.colab import userdata

with open(os.path.join(PROJECT_FOLDER, "notion_config.json"), "r") as f:
    NOTION_CONFIG = json.load(f)

try:
    publisher = NotionPublisher(
        token=userdata.get("NOTION_TOKEN"),
        databases_config=NOTION_CONFIG
    )
    
    print(f"🚀 Iniciando publicação da Sessão {SESSION_NUMBER}...")
    page_id = publisher.publish_session(ID_MESA, SESSION_NUMBER, SESSION_DATE, CHRONICLE_OUTPUT_FILE)
    print(f"✅ Sucesso! Página criada: https://notion.so/{page_id.replace('-', '')}")

except Exception as e:
    print(f"❌ Erro na integração com Notion: {e}")